# NGC 4151 spectral fit with the full DC4 source catalog

Use the full-observation YAML and matching response JSON imported from
`DC4-analysis`. The catalog contains 71 simulation entries (52 point sources and
19 extended sources). Fit NGC 4151 with a **cutoff power law**: K, index and cutoff
are free, pivot and position are fixed. All 70 other sources remain fixed; the
instrumental-background rate is free. The injected reference is CPL + PL.

No catalog-generation notebook needs to be run first. The setup below creates
missing extended-source spatial FITS templates from the original `cosi-sim`
maps and saves them for reuse. It loads or generates one NGC 4151 GTI extended
response (about 13 GiB in memory). This is separate from the much smaller
spatial templates. Allow additional RAM for the fit and response construction.

Reusable helpers are provided by this cosipy checkout; no adjacent Python
files are required. Set the paths below for your machine. The full YAML and JSON are never rewritten; paths are relocated in
memory. The original source-library spectra, maps, and light curves are required.

The YAML amplitudes already include the full-observation temporal mean. Responses
for variable sources use **L(t) / mean_full(L)** over the analysis GTI; do not
multiply the source amplitudes by another duration factor. The recipe's lightcurve
hashes are checked. Its nova 500-chunk convention is an inferred simulation
assumption, and previously observed model/count discrepancies are not calibrated
away by this fit. Unpolarized folding matches the catalog-generation workflow.

The supplied mock includes all sources and an updated NGC 4151 injection. It is
not an isolated-source dataset. Other sources are modeled, not subtracted from
the observed counts or added to the instrumental-background template.

Run from the top in a fresh kernel. Inspect fit convergence, parameter boundaries,
and covariance-sample rejection before interpreting the plotted 68% interval.

In [ ]:
from pathlib import Path
from copy import deepcopy
import sys

import astropy.units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from astromodels import (
    Cutoff_powerlaw, ExtendedSource, PointSource, Parameter, Powerlaw, Model, load_model,
)
from histpy import Histogram
from threeML import DataList, JointLikelihood

from cosipy import BinnedData
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.data_io import EmCDSBinnedData
from cosipy.event_selection import GoodTimeInterval
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response import (
    BinnedInstrumentResponse,
    BinnedThreeMLModelFolding,
    BinnedThreeMLPointSourceResponse,
    BinnedThreeMLExtendedSourceResponse,
    ExtendedSourceResponse,
)
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.statistics import PoissonLikelihood
from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat  # noqa: F401

%matplotlib inline

## Tutorial setup utilities

The following definition cells interpret the DC4 recipe, map its source names
to original source-library assets, and load/select DC4 event files. They do not
run the fit or write files until called by the configuration cell below.

Lightcurve integration, weighted/component response adapters, and spatial-map
conversion are imported from cosipy. No adjacent Python helper files are needed.
These DC4-specific definitions can be left unchanged for this tutorial.


In [ ]:
# DC4 recipe interpretation stays in this tutorial; reusable calculations live in cosipy.
import json
import yaml
from astropy.io import fits
from cosipy.response.temporal_profile import LightcurveProfile
from cosipy.response.threeml_temporal_response import (
    ZeroPointSourceResponse, ComponentPointSourceResponse,
    make_weighted_point_source_response,
)
from cosipy.util.megalib import megalib_map_to_galprop_fits


In [ ]:
def load_recipe(path, catalog_path, model):
    # A relocated recipe may be supplied in memory; retain the same validation.
    recipe = deepcopy(path) if isinstance(path, dict) else json.loads(Path(path).read_text())
    if recipe.get('schema') != 'dc4-component-lightcurve-response-v1':
        raise ValueError("Unsupported temporal-response recipe schema")
    if Path(recipe['catalog']).resolve() != Path(catalog_path).resolve():
        raise ValueError("The response recipe belongs to a different catalog")
    if recipe['selection'] not in {'full', 'NGC4151_GTI'} or recipe['time_basis'] != 'elapsed':
        raise ValueError("Expected a full-observation or NGC4151 GTI elapsed-time catalog")
    if set(row['source'] for row in recipe['components']) != set(model.sources):
        raise ValueError("Recipe and catalog source lists differ")
    if any(row['selection'] != recipe['selection'] for row in recipe['components']):
        raise ValueError("Mixed selections in response recipe")
    keys = [(r['source'], r['component']) for r in recipe['components']]
    if len(set(keys)) != len(keys):
        raise ValueError("Duplicate source/component recipe entries")
    return recipe


def component_name(source, label):
    if label in source.components:
        return label
    # v1 recipes store MEGAlib labels rather than astromodels component names.
    crab_names = {'Crab_Nebula': 'nebula', 'Crab_P1': 'peak1',
                  'Crab_Brg': 'bridge', 'Crab_P2': 'peak2'}
    if source.name == 'crab' and crab_names.get(label) in source.components:
        return crab_names[label]
    raise ValueError(f"Unknown temporal component {source.name}/{label}")


def make_temporal_responses(model, recipe, base_history, gti, data,
                            instrument_response, detector_response,
                            exposure_multiplier=1., full_history=None):
    """Build per-source/component responses from the unscaled GTI history.

    Validate temporal factors over the CATALOG selection, which can differ
    from the analysis GTI. Full catalogs require the original full_history:
    response weights remain L(t)/mean_full while exposure is GTI restricted.
    Exposure repetitions multiply response livetimes, not fluxes.
    Responses are computed lazily by cosipy and reused throughout fitting.
    """
    if not np.isfinite(exposure_multiplier) or exposure_multiplier <= 0:
        raise ValueError("Exposure multiplier must be finite and positive")
    starts, stops = np.asarray(gti.tstart_list.unix), np.asarray(gti.tstop_list.unix)
    duration = float(np.sum(stops-starts))
    if duration <= 0 or np.any(stops <= starts):
        raise ValueError("The analysis GTI must have positive duration")
    catalog_selection = recipe.get('selection', 'NGC4151_GTI')
    if catalog_selection == 'full':
        if full_history is None:
            raise ValueError("Full-observation catalogs require the original full_history")
        reference_starts = np.array([full_history.obstime[0].unix])
        reference_stops = np.array([full_history.obstime[-1].unix])
        if np.any(starts < reference_starts[0]-1e-5) or np.any(stops > reference_stops[0]+1e-5):
            raise ValueError("The analysis GTI extends beyond the catalog observation")
    elif catalog_selection == 'NGC4151_GTI':
        reference_starts, reference_stops = starts, stops
    else:
        raise ValueError(f"Unsupported catalog selection: {catalog_selection}")
    reference_duration = float(np.sum(reference_stops-reference_starts))
    if not np.isfinite(reference_duration) or reference_duration <= 0:
        raise ValueError("Invalid catalog reference duration")
    history_starts, history_stops = base_history.intervals_tstart.unix, base_history.intervals_tstop.unix
    rows_by_source = {}
    for row in recipe['components']:
        rows_by_source.setdefault(row['source'], []).append(row)
    overrides, diagnostics = {}, []
    for key, rows in rows_by_source.items():
        source = model.sources[key]
        for row in rows:
            if not np.isclose(row['selected seconds'], reference_duration, rtol=0., atol=1e-4):
                raise ValueError(f"Reference duration differs from the catalog: {key}")
        if not any(row['lightcurve'] for row in rows):
            continue
        if not isinstance(source, PointSource):
            raise NotImplementedError(f"A temporal extended response is required for {key}")
        if any(r['component'] == 'all' for r in rows) and len(rows) != 1:
            raise ValueError(f"Invalid component grouping for {key}")
        component_responses = {}
        for row in rows:
            applied = float(row['applied factor'])
            if not np.isfinite(applied) or applied <= 0:
                raise ValueError(f"Invalid applied factor for {key}")
            if row['lightcurve']:
                profile = LightcurveProfile.from_dat(
                    row['lightcurve'], expected_sha256=row['lightcurve sha256'],
                    repeating=row['repeating'], mode=row['normalization mode'],
                    bins=int(row['simulation bins']),
                )
                mean = float(profile.integral(reference_starts, reference_stops).sum()/reference_duration)
                gti_mean = float(profile.integral(starts, stops).sum()/duration)
                weights = profile.response_weights(history_starts, history_stops, applied)
                del profile
            else:
                mean, gti_mean, weights = 1., 1., np.ones(base_history.nintervals)
            if not np.isclose(mean, row['time factor'], rtol=1e-8, atol=1e-14):
                raise ValueError(f"Temporal factor differs from catalog for {key}/{row['component']}")
            if not np.isclose(applied, max(mean, 1e-30), rtol=1e-8, atol=0.):
                raise ValueError(f"Applied factor differs from catalog for {key}")
            weighted_live = base_history.livetime * weights * exposure_multiplier
            response = make_weighted_point_source_response(
                base_history, weights, data, instrument_response, detector_response,
                exposure_multiplier=exposure_multiplier,
            )
            diagnostics.append({'source': key, 'component': row['component'],
                                'normalization mode': row['normalization mode'],
                                'simulation bins': row['simulation bins'],
                                'catalog selection': catalog_selection,
                                'catalog time factor': mean,
                                'analysis GTI time factor': gti_mean,
                                'response denominator': applied,
                                'weighted response seconds': float(weighted_live.to_value(u.s).sum())})
            if row['component'] == 'all':
                overrides[key] = response
            else:
                name = component_name(source, row['component'])
                if name in component_responses:
                    raise ValueError(f"Duplicate mapped component for {key}/{name}")
                component_responses[name] = response
        if component_responses:
            if set(component_responses) != set(source.components):
                raise ValueError(f"Recipe does not cover all components of {key}")
            overrides[key] = ComponentPointSourceResponse(component_responses)
    return overrides, diagnostics


In [ ]:
SPATIAL_TEMPLATE_NSIDE = 8
SPATIAL_TEMPLATE_FORMAT_TAG = "perMeV_v3"
SPATIAL_DEFINITIONS = {'positrons_26al_line': ('DC3', 'Positrons_from_26Al_line'),
 'positrons_26al_cont': ('DC3', 'Positrons_from_26Al_cont'),
 'positrons_44ti_line': ('DC3', 'Positrons_from_44Ti_line'),
 'positrons_44ti_cont': ('DC3', 'Positrons_from_44Ti_cont'),
 'narrow_bulge_511': ('DC3', 'NB_511'),
 'broad_bulge_511': ('DC3', 'BB_511'),
 'vela_snr_511': ('DC3', 'Vela_SNR_511'),
 'al26_ne2001': ('DC3', '26Al_NE2001'),
 'fe60_ne2001': ('DC3', '60Fe_NE2001'),
 'positrons_thin_disk_cont': ('DC4', 'Positrons_Thin_Disk_cont'),
 'positrons_thin_disk_line': ('DC4', 'Positrons_Thin_Disk_line')}

def resolve_source_asset(source_definition, filename):
    path = source_definition.parent / filename
    candidates = [path, Path(f"{path}.gz")]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not resolve {filename!r} next to {source_definition}"
    )


def parse_megalib_blocks(source_definition):
    # Return one property dictionary per DataChallenge.Source block.
    blocks = []
    by_megalib_name = {}
    for raw_line in source_definition.read_text().splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        tokens = line.split()
        if tokens[:1] == ["DataChallenge.Source"]:
            megalib_name = tokens[1]
            block = {"_name": megalib_name}
            blocks.append(block)
            by_megalib_name[megalib_name] = block
            continue
        if "." not in tokens[0]:
            continue
        megalib_name, prop = tokens[0].split(".", 1)
        if megalib_name in by_megalib_name:
            by_megalib_name[megalib_name][prop] = tokens[1:]
    if not blocks:
        raise ValueError(f"No DataChallenge.Source blocks in {source_definition}")
    return blocks


def source_library_path(value, source_library):
    """Relocate a catalog asset without relying on the original user's home."""
    parts = Path(value).parts
    if "Source_Library" not in parts:
        raise ValueError(f"Not a source-library path: {value}")
    relative = Path(*parts[parts.index("Source_Library") + 1:])
    candidate = (Path(source_library) / relative).resolve()
    if not candidate.is_file():
        raise FileNotFoundError(candidate)
    return str(candidate)


def ensure_spatial_template(source_name, output_path, source_library, overwrite=False):
    """Build only missing templates, preserving the generator's conversion."""
    output_path = Path(output_path)
    if output_path.exists() and not overwrite:
        with fits.open(output_path) as hdul:
            if (hdul['SKYMAP'].header.get('NSIDE') != SPATIAL_TEMPLATE_NSIDE
                    or hdul['SKYMAP'].header.get('MAPVERS') != SPATIAL_TEMPLATE_FORMAT_TAG):
                raise ValueError(f"Incompatible cached spatial template: {output_path}")
        return output_path, 'loaded'
    challenge, stem = SPATIAL_DEFINITIONS[source_name]
    matches = sorted((Path(source_library) / challenge / 'sources').rglob(f'{stem}.source'))
    if len(matches) != 1:
        raise ValueError(f"Expected one {stem}.source, found {matches}")
    blocks = parse_megalib_blocks(matches[0])
    if len(blocks) != 1 or len(blocks[0].get('Beam', [])) < 2:
        raise ValueError(f"Expected one coupled map in {matches[0]}")
    map_path = resolve_source_asset(matches[0], blocks[0]['Beam'][1])
    megalib_map_to_galprop_fits(
        map_path, output_path, nside=SPATIAL_TEMPLATE_NSIDE,
        oversample_nside=64, energy_bounds=(100., 10000.), overwrite=overwrite,
    )
    return output_path, 'created'


def prepare_catalog(catalog_path, recipe_path, source_library, template_directory,
                    orientation_path, response_path, binning_path, overwrite=False):
    """Generate templates, relocate external assets, and load without rewriting YAML.

    YAML/JSON supplied from DC4-analysis remain byte-for-byte unchanged. Only
    the in-memory dictionaries receive this computer's paths. Their source
    amplitudes and the recipe's light-curve hashes are never modified.
    """
    from astromodels.core.model_parser import ModelParser
    from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat  # noqa: F401

    catalog_path, recipe_path = Path(catalog_path), Path(recipe_path)
    contents = yaml.safe_load(catalog_path.read_text())
    recipe = json.loads(recipe_path.read_text())
    if Path(recipe['catalog']).name != catalog_path.name:
        raise ValueError('The recipe refers to a different catalog filename')
    recipe['catalog'] = str(catalog_path.resolve())
    for key, local in [('orientation', orientation_path), ('detector_response', response_path),
                       ('binning', binning_path)]:
        if Path(recipe[key]).name != Path(local).name:
            raise ValueError(f'{key} filename differs from the catalog recipe')
        recipe[key] = str(Path(local).resolve())
    for row in recipe['components']:
        if row['lightcurve']:
            row['lightcurve'] = source_library_path(row['lightcurve'], source_library)

    template_rows = []
    def relocate(node, source_name):
        if isinstance(node, dict):
            for key, value in node.items():
                if key == '_fitsfile' and value:
                    expected_name = f'{source_name}_nside8_perMeV_v3.fits'
                    if Path(value).name != expected_name:
                        raise ValueError(f'Unsupported spatial template: {value}')
                    path, status = ensure_spatial_template(
                        source_name, Path(template_directory) / expected_name,
                        source_library, overwrite=overwrite,
                    )
                    node[key] = str(path.resolve())
                    template_rows.append({'source': source_name, 'status': status, 'file': str(path)})
                elif isinstance(value, str) and 'Source_Library/' in value:
                    node[key] = source_library_path(value, source_library)
                else:
                    relocate(value, source_name)
        elif isinstance(node, list):
            for value in node:
                relocate(value, source_name)
    for source_key, definition in contents.items():
        relocate(definition, source_key.split(' (')[0])
    model = ModelParser(model_dict=contents).get_model()
    recipe = load_recipe(recipe, catalog_path, model)
    if recipe['selection'] != 'full':
        raise ValueError('This tutorial requires the full-observation catalog')
    return model, recipe, template_rows


In [ ]:
def load_selected_histogram(path, gti, binning_path, already_selected=True):
    """Load explicitly preselected HDF5 or select native FITS events before binning.

    Histogram axes do not encode their GTI. For an already-selected histogram,
    its time-selection provenance is the caller's responsibility. An unselected
    projected histogram is rejected because event times cannot be recovered.
    Native event selection uses half-open [start, stop) GTI intervals.
    """
    import astropy.units as u
    from astropy.coordinates import SkyCoord
    from histpy import Histogram, Axis, Axes, HealpixAxis

    config = yaml.safe_load(Path(binning_path).read_text())
    axes = Axes([
        Axis(config['energy_bins'], unit=u.keV, label='Em'),
        Axis(np.linspace(0., 180., int(180. / config['phi_pix_size']) + 1),
             unit=u.deg, label='Phi'),
        HealpixAxis(nside=config['nside'], scheme=config['scheme'],
                    coordsys='galactic', label='PsiChi'),
    ])
    path = Path(path)
    is_fits = path.name.endswith(('.fits', '.fits.gz'))
    if not is_fits:
        if not already_selected:
            raise ValueError('Use native event FITS to apply a new GTI, not a projected histogram')
        histogram = Histogram.open(path).project('Em', 'Phi', 'PsiChi')
        if histogram.axes != axes:
            raise ValueError(f'Histogram binning differs from agn.yaml: {path}')
        return histogram

    starts = np.asarray(gti.tstart_list.unix, dtype=float)
    stops = np.asarray(gti.tstop_list.unix, dtype=float)
    if (starts.size == 0 or np.any(stops <= starts)
            or np.any(starts[1:] < stops[:-1])):
        raise ValueError('GTI intervals must be positive, sorted and nonoverlapping')
    histogram = Histogram(axes, sparse=True)
    with fits.open(path, memmap=False) as hdul:
        events = hdul[1].data
        for offset in range(0, len(events), 250_000):
            block = events[offset:offset + 250_000]
            times = np.asarray(block['TimeTags'], dtype=float)
            if already_selected:
                selected = np.ones(times.size, dtype=bool)
            else:
                index = np.searchsorted(starts, times, side='right') - 1
                selected = (index >= 0) & (times < stops[np.maximum(index, 0)])
            direction = SkyCoord(
                l=np.asarray(block['Chi galactic'][selected]) * u.deg,
                b=np.asarray(block['Psi galactic'][selected]) * u.deg,
                frame='galactic',
            )
            histogram.fill(np.asarray(block['Energies'][selected]) * u.keV,
                           np.asarray(block['Phi'][selected]) * u.rad, direction)
    return histogram


## 1. Configure inputs and caches

The default input histograms already have the NGC 4151 60-degree
pointing/occultation time selection applied. They must correspond to the GTI
computed below. The notebook does **not** apply a second cut to those histograms.
Alternatively, specify native event FITS inputs and set the corresponding
`*_already_gti_selected` flag to False: their event times are selected by the
computed GTI before binning. Full-observation histograms without an event-time
axis cannot be accurately reselected and must not be substituted for cut data.

Spatial templates and extended responses are cached without overwriting existing
files. Changing the selection, detector response or binning requires a different
extended-response cache; the default cache is specifically for NGC 4151 at 60°.

In [ ]:
# Run Jupyter from this folder or anywhere inside the cosipy checkout.
notebook_dir = Path.cwd().resolve()
if not (notebook_dir / "source_catalog_DC4_all_71files_full_elapsed_lightcurve_response_norm.yaml").is_file():
    roots = [p for p in (notebook_dir, *notebook_dir.parents) if (p / "cosipy").is_dir()]
    if not roots:
        raise FileNotFoundError("Run from the cosipy checkout or set notebook_dir explicitly")
    notebook_dir = roots[0] / "docs/tutorials/spectral_fits/continuum_fit/AGN"

# Edit these two locations for your machine.
dc4_files_root = Path.home() / (
    "Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files"
)
source_library = Path.home() / "Software/cosi-sim/cosi_sim/Source_Library"
mock_data_path = dc4_files_root / (
    "Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p30_time_cut.hdf5"
)
background_path = dc4_files_root / (
    "Background/Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC4151_60deg_fov_cut.hdf5"
)
mock_already_gti_selected = True
background_already_gti_selected = True
orientation_path = dc4_files_root / "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
response_path = dc4_files_root / (
    "ResponseContinuum.o3.e100_10000.b10log.s10396905069491."
    "m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
)
agn_config = notebook_dir / "agn.yaml"
source_catalog_path = notebook_dir / "source_catalog_DC4_all_71files_full_elapsed_lightcurve_response_norm.yaml"
response_recipe_path = source_catalog_path.with_suffix(".response.json")

# Small, reusable source-model files, generated by this notebook if absent.
spatial_template_directory = dc4_files_root / "DC4_Sources/DC4_Spatial_Templates"
overwrite_spatial_templates = False
# Large GTI-dependent instrument response, not the source-model templates above.
extended_response_path = dc4_files_root / (
    "Extended_Responses/NGC4151_Cut/"
    "DC4_continuum_NGC4151_60deg_GTI_extended_response_nside8.h5"
)

# Counts and response exposure are scaled together; source fluxes are unchanged.
# 8 means eight repeats of this observation, NOT a new Poisson realization.
exposure_multiplier = 1
exposure_months = 3 * exposure_multiplier
if not np.isfinite(exposure_multiplier) or exposure_multiplier <= 0:
    raise ValueError("exposure_multiplier must be finite and positive")
for path in (mock_data_path, background_path, orientation_path, response_path,
             agn_config, source_catalog_path, response_recipe_path, source_library):
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}. Update the paths above.")

# Generate missing spatial templates, then load the relocated catalog in memory.
model, recipe, template_status = prepare_catalog(
    source_catalog_path, response_recipe_path, source_library,
    spatial_template_directory, orientation_path, response_path, agn_config,
    overwrite=overwrite_spatial_templates,
)
display(pd.DataFrame(template_status))
assert len(model.sources) == 71 and "ngc4151" in model.point_sources
assert not model.free_parameters
print(f"Catalog: {len(model.point_sources)} point + {len(model.extended_sources)} extended sources")
print("Temporal assumptions:", recipe["normalization_assumptions"])

## 2. Compute the NGC 4151 GTI

Use the 60° pointing cut with Earth occultation, matching the supplied cut data
and background. The full catalog's JSON `gti_target` describes the generator's
other output, not the selection of its full-observation normalizations.

Retain the full history for normalization verification and the GTI history for
response generation. Variable-source weights keep the full-observation mean
in the denominator. Sources that emit entirely outside this GTI predict zero
counts even when their full-observation catalog amplitude is nonzero.

In [ ]:
source_l = model.ngc4151.position.l.value * u.deg
source_b = model.ngc4151.position.b.value * u.deg
source_coord = SkyCoord(l=source_l, b=source_b, frame="galactic")

# These settings belong to the selected data, not to the full catalog.
max_offaxis = 60.0 * u.deg
earth_occ = True

sc_orientation_full = SpacecraftHistory.open(orientation_path)
source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    sc_orientation_full,
    max_offaxis,
    earth_occ=earth_occ,
)
sc_orientation_cut = sc_orientation_full.apply_gti(source_gti)

# Repeat the selected pointing history by scaling its interval livetimes.
# The attitude and Earth-occultation selection remain unchanged.
sc_orientation = SpacecraftHistory(
    sc_orientation_cut.obstime,
    sc_orientation_cut.attitude,
    sc_orientation_cut.location,
    livetime=sc_orientation_cut.livetime * exposure_multiplier,
)
sc_orientation.cache_earth_occ = sc_orientation_cut.cache_earth_occ


full_livetime = sc_orientation_full.cumulative_livetime().to_value(u.s)
fov_livetime = sc_orientation_cut.cumulative_livetime().to_value(u.s)
scaled_livetime = sc_orientation.cumulative_livetime().to_value(u.s)
print(f"Full orientation livetime: {full_livetime:,.1f} s")
print(f"Three-month GTI livetime: {fov_livetime:,.1f} s")
print(f"Scaled {exposure_months}-month livetime: {scaled_livetime:,.1f} s")

## 3. Load or select the data and background

Use identical binning and GTI for both. Native event FITS inputs are time-selected
and binned here; the default already-cut HDF5 inputs are simply loaded. No native
individual-source event files are required. A machine-tiny background floor is
retained from the reference workflow; it is numerical regularization, not a
measurement of the background in empty bins.

In [ ]:
data_hist = load_selected_histogram(
    mock_data_path, source_gti, agn_config,
    already_selected=mock_already_gti_selected,
)
total_bkg = load_selected_histogram(
    background_path, source_gti, agn_config,
    already_selected=background_already_gti_selected,
)
if total_bkg.axes != data_hist.axes:
    raise ValueError("Mock data and background must have identical Em/Phi/PsiChi axes")
data_hist *= exposure_multiplier
total_bkg *= exposure_multiplier
total_bkg += sys.float_info.min
bkg_dist = {"total_bkg": total_bkg}
background_rate_initial = float(total_bkg.to_dense(copy=False).contents.sum()) / scaled_livetime
if not np.isfinite(background_rate_initial) or background_rate_initial <= 0:
    raise ValueError("Invalid initial instrumental-background rate")
print(f"Initial instrumental-background rate: {background_rate_initial:.6g} Hz")

Open the continuum response.

In [ ]:
dr = FullDetectorResponse.open(str(response_path))

## 4. Construct the responses and fit

Construct the ordinary point-source response and a single ordinary extended
response for the GTI. Generate and save the extended response only if absent;
otherwise reuse it. Exposure scaling is applied in memory after saving/loading.

Variable point sources receive their source/component-specific lightcurve
responses. Crab's components remain inside one catalog source. The YAML amplitudes
are unchanged. Use unpolarized folding consistently with the catalog generator.

In [ ]:
# Match the generator's unpolarized response approximation.
unpolarized_components = []
for source in model.sources.values():
    for component in source.components.values():
        degree = getattr(component.polarization, "degree", None)
        if degree is not None and degree.value != 0.0:
            degree.value = 0.0
            unpolarized_components.append(f"{source.name}.{component.name}")
print("Using unpolarized folding for:", unpolarized_components)

data = EmCDSBinnedData(data_hist)
bkg = FreeNormBinnedBackground(bkg_dist, sc_history=sc_orientation, copy=False)
instrument_response = BinnedInstrumentResponse(dr, data)
psr = BinnedThreeMLPointSourceResponse(
    data=data,
    instrument_response=instrument_response,
    sc_history=sc_orientation,
    energy_axis=dr.axes["Ei"],
    polarization_axis=dr.axes["Pol"] if "Pol" in dr.axes.labels else None,
    nside=2 * data.axes["PsiChi"].nside,
)

# Drop references to the previous response if this setup cell is rerun.
for name in ("like", "plugins", "cosi", "like_fun", "response", "esr", "extended_response", "source_specific_responses"):
    globals().pop(name, None)
if extended_response_path.exists():
    print("Loading cached GTI extended response (about 13 GiB)...")
    extended_response = ExtendedSourceResponse.open(extended_response_path)
else:
    print("Generating the NGC 4151 GTI extended response; this can be slow...")
    # Cache the unscaled three-month GTI response, never an exposure-multiplied one.
    extended_response = dr.get_extended_source_response(
        sc_orientation_cut, coordsys="galactic", nside_image=8,
        nside_scatt_map=2 * data.axes["PsiChi"].nside, earth_occ=earth_occ,
    )
    extended_response_path.parent.mkdir(parents=True, exist_ok=True)
    extended_response.write(extended_response_path, overwrite=False)
    print("Saved:", extended_response_path)
if extended_response.axes["NuLambda"].nside != 8:
    raise ValueError("Expected the catalog generator's NSIDE-8 extended response")
if extended_response.axes[2:] != data.axes:
    raise ValueError("Extended response and data Em/Phi/PsiChi axes do not match")
if extended_response.axes["Ei"] != dr.axes["Ei"]:
    raise ValueError("Point and extended responses have different incident-energy bins")
extended_response *= exposure_multiplier
esr = BinnedThreeMLExtendedSourceResponse(
    data=data,
    precomputed_psr=extended_response,
)
source_specific_responses, temporal_diagnostics = make_temporal_responses(
    model, recipe, sc_orientation_cut, source_gti, data,
    instrument_response, dr, exposure_multiplier=exposure_multiplier,
    full_history=sc_orientation_full,
)
display(pd.DataFrame(temporal_diagnostics))
response = BinnedThreeMLModelFolding(
    data=data,
    point_source_response=psr,
    extended_source_response=esr,
    source_specific_point_source_responses=source_specific_responses,
)
like_fun = PoissonLikelihood(data, response, bkg)
cosi = ThreeMLPluginInterface("cosi", like_fun, response, bkg)
cosi.bkg_parameter["total_bkg"] = Parameter(
    "total_bkg",
    background_rate_initial,
    min_value=0.0,
    max_value=50.0,
    delta=max(0.05 * background_rate_initial, 1e-6),
    unit=u.Hz,
)
print(f"Mixed response ready for all {len(model.sources)} catalog sources.")
print("Temporal point-source overrides:", sorted(source_specific_responses))
print("Full-observation amplitudes are unchanged; responses integrate only over the NGC 4151 GTI.")
print("Response denominator = catalog full-observation mean, not the GTI mean.")

Replace only NGC 4151's in-memory catalog spectrum with a cutoff power
law. Fit its normalization `K`, photon index `index`, and cutoff energy `xc`.
The pivot remains fixed at 200 keV: freeing it together with `K` would create
a redundant normalization parameter. The source position and all 70 nuisance
sources remain fixed, and the YAML is not overwritten.

The injected reference remains the original CPL + PL used to generate the
mock data. Thus the comparison plot shows a **CPL-only fit** against the
**injected total (CPL + PL)**, not a changed simulation.

In [ ]:
D = 0.30

# Injected spectrum used only for comparison in the plot.
spectrum_inj_cpl = Cutoff_powerlaw()
spectrum_inj_cpl.K.value = 0.15
spectrum_inj_cpl.piv.value = 1.0
spectrum_inj_cpl.xc.value = 200.0
spectrum_inj_cpl.index.value = -1.75
spectrum_inj_cpl.K.unit = 1 / (u.cm**2 * u.s * u.keV)
spectrum_inj_cpl.piv.unit = u.keV
spectrum_inj_cpl.xc.unit = u.keV

spectrum_inj_tail = Powerlaw()
spectrum_inj_tail.K.value = (
    D * spectrum_inj_cpl.evaluate_at(200.0)
)
spectrum_inj_tail.piv.value = 200.0
spectrum_inj_tail.index.value = -3.8
spectrum_inj_tail.K.unit = 1 / (u.cm**2 * u.s * u.keV)
spectrum_inj_tail.piv.unit = u.keV
spectrum_inj = spectrum_inj_cpl + spectrum_inj_tail

# Replace NGC 4151 only; initialize from the catalog's thermal component.
assert len(model.free_parameters) == 0
catalog_source = model.ngc4151
catalog_shape = catalog_source.spectrum.main.composite
spectrum_cpl = Cutoff_powerlaw(
    K=catalog_shape.K_1.value,
    piv=catalog_shape.piv_1.value,
    index=catalog_shape.index_1.value,
    xc=catalog_shape.xc_1.value,
)
ngc4151_cpl_source = PointSource(
    "ngc4151",
    l=catalog_source.position.l.value,
    b=catalog_source.position.b.value,
    spectral_shape=spectrum_cpl,
)
for parameter in ngc4151_cpl_source.parameters.values():
    parameter.fix = True
model.remove_source("ngc4151")
model.add_source(ngc4151_cpl_source)
ngc4151_fit = model.ngc4151.spectrum.main.Cutoff_powerlaw

ngc4151_fit.K.min_value = 1e-8
ngc4151_fit.K.max_value = 1e-2
ngc4151_fit.K.delta = 0.1 * ngc4151_fit.K.value
ngc4151_fit.K.fix = False

ngc4151_fit.index.min_value = -10.0
ngc4151_fit.index.max_value = 10.0
ngc4151_fit.index.delta = 0.1
ngc4151_fit.index.fix = False

ngc4151_fit.xc.min_value = 100.0
ngc4151_fit.xc.max_value = 10000.0
ngc4151_fit.xc.delta = 0.1 * ngc4151_fit.xc.value
ngc4151_fit.xc.fix = False
ngc4151_fit.piv.fix = True

expected_free_parameters = {
    "ngc4151.spectrum.main.Cutoff_powerlaw.K",
    "ngc4151.spectrum.main.Cutoff_powerlaw.index",
    "ngc4151.spectrum.main.Cutoff_powerlaw.xc",
}
assert set(model.free_parameters) == expected_free_parameters
assert len(model.sources) == 71
print("NGC 4151 fit model: cutoff power law only")
print("Free parameters:", list(model.free_parameters))

Fit the CPL source model and instrumental-background normalization once.
The first likelihood evaluation builds the source responses and can be slow;
subsequent evaluations reuse them. All other catalog parameters remain fixed.


In [ ]:
plugins = DataList(cosi)
like = JointLikelihood(model, plugins, verbose=False)
_ = like.fit()

## Error propagation and plotting

Propagate the fitted parameter uncertainties to the NGC 4151 spectrum.

In [ ]:
results = like.results
print(results.display())

optimized_model = results.optimized_model
fitted_shape = optimized_model.ngc4151.spectrum.main.Cutoff_powerlaw

# Propagate all three correlated CPL parameters without mutating the fit model.
def ngc4151_flux(e, K, index, xc):
    return K * (e / fitted_shape.piv.value)**index * np.exp(-e / xc)

results_err = results.propagate(
    ngc4151_flux,
    K=results.get_variates(fitted_shape.K.path),
    index=results.get_variates(fitted_shape.index.path),
    xc=results.get_variates(fitted_shape.xc.path),
)
print(optimized_model.ngc4151)

Evaluate the fitted and injected spectra from 200 keV to 5 MeV.

In [ ]:
energy = np.geomspace(200 * u.keV, 5 * u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux = results_err(e)
    flux_median[i] = fitted_shape.evaluate_at(e)
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)
    flux_inj[i] = spectrum_inj.evaluate_at(e)

binned_energy_edges = data_hist.axes["Em"].edges.to_value(u.keV)
binned_energy = 0.5 * (binned_energy_edges[1:] + binned_energy_edges[:-1])
like.restore_best_fit()
_ = cosi.get_log_like()  # Synchronize the restored nuisance/background rate.
expectation = response.expectation()

Compare the best-fit CPL spectrum with the injected CPL+PL total.
The shaded region is the propagated 68% interval; it is conditional on the fixed
catalog models. The following count-space projection includes every source plus
the fitted instrumental background. The likelihood uses the full Em/Phi/PsiChi space.

In [ ]:
FONT_SIZE = 20
plt.rcParams["agg.path.chunksize"] = 10000
plt.rcParams.update({"font.size": FONT_SIZE})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams.update({
    "font.weight": "normal",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
})


def style_axis(ax):
    ax.xaxis.set_tick_params(which="major", size=12, width=1.5, direction="in", top=True, pad=8, labelsize=FONT_SIZE)
    ax.xaxis.set_tick_params(which="minor", size=6, width=1.5, direction="in", top=True, pad=8)
    ax.yaxis.set_tick_params(which="major", size=12, width=1.5, direction="in", right=True, pad=8, labelsize=FONT_SIZE)
    ax.yaxis.set_tick_params(which="minor", size=6, width=1.5, direction="in", right=True, pad=8)
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)


fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
style_axis(ax)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.plot(
    energy / 1000,
    energy * energy * flux_inj,
    color="black",
    ls=":",
    lw=3,
    label="Injected total (CPL + PL)",
)
ax.plot(
    energy / 1000,
    energy * energy * flux_median,
    color="red",
    lw=2,
    linestyle="dashdot",
    label="Best fit (CPL; full catalog)",
)
ax.fill_between(
    energy / 1000,
    energy * energy * flux_lo,
    energy * energy * flux_hi,
    alpha=0.18,
    color="red",
    label="Catalog-aware: 68% interval",
)

ax.set_xscale("log")
ax.set_yscale("log")
# Include the entire fitted uncertainty band in the plotted range.
plotted_energy_flux = np.concatenate([
    energy * energy * flux_inj, energy * energy * flux_hi,
])
finite_positive_flux = plotted_energy_flux[np.isfinite(plotted_energy_flux) & (plotted_energy_flux > 0)]
ax.set_ylim(1e-4, max(10., 1.5 * finite_positive_flux.max()))
ax.xaxis.set_major_locator(mticker.FixedLocator([0.2, 1.0, 5.0]))
ax.xaxis.set_major_formatter(mticker.FixedFormatter(["0.2", "1.0", "5.0"]))
ax.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE)
ax.set_ylabel(r"Energy Flux (keV cm$^{-2}$ s$^{-1}$)", fontsize=FONT_SIZE)

thermal_flux = np.asarray([spectrum_inj_cpl.evaluate_at(e) for e in energy])
tail_flux = np.asarray([spectrum_inj_tail.evaluate_at(e) for e in energy])
thermal_energy_flux = np.trapezoid(energy * thermal_flux, energy)
tail_energy_flux = np.trapezoid(energy * tail_flux, energy)
energy_flux_ratio = tail_energy_flux / (thermal_energy_flux + tail_energy_flux)

ax.text(
    0.97,
    0.97,
    f"NGC 4151\n{exposure_months}-months\nCPL-only fit\nInjected NT Flux Fraction = {energy_flux_ratio:.2f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=FONT_SIZE,
    fontweight="normal",
)
ax.legend(fontsize=17, loc="lower left", frameon=False)
plt.show()

Compare the complete temporally folded 71-entry catalog plus the fitted
instrumental background with the observed mock counts. This is a projection
onto measured energy; the likelihood itself fits the full Em/Phi/PsiChi space.


In [ ]:
expectation_bkg = bkg.expectation(copy=True)
model_counts = (
    expectation.project("Em").to_dense(copy=False).contents
    + expectation_bkg.project("Em").to_dense(copy=False).contents
)
data_counts = data.data.project("Em").to_dense(copy=False).contents

fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
style_axis(ax)

ax.stairs(
    model_counts,
    binned_energy_edges / 1000,
    color="purple",
    label="Best fit plus background",
)
ax.errorbar(
    binned_energy / 1000,
    model_counts,
    yerr=np.sqrt(model_counts),
    color="purple",
    linewidth=0,
    elinewidth=1.5,
)
ax.stairs(
    data_counts,
    binned_energy_edges / 1000,
    color="black",
    ls=":",
    lw=2,
    label="Observed counts",
)
ax.errorbar(
    binned_energy / 1000,
    data_counts,
    yerr=np.sqrt(data_counts),
    color="black",
    linewidth=0,
    elinewidth=1.5,
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.xaxis.set_major_locator(mticker.FixedLocator([0.2, 1.0, 5.0]))
ax.xaxis.set_major_formatter(mticker.FixedFormatter(["0.2", "1.0", "5.0"]))
ax.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE)
ax.set_ylabel("Counts", fontsize=FONT_SIZE)
ax.legend(fontsize=23, loc="lower left", frameon=False)
plt.show()